# GRPO Fine-Tuning Simulation

Colab notebook for a small GRPO run inspired by AMD ROCm Blog: Fine-Tuning LLMs with GRPO on AMD MI300X.

## 1. Clone your GitHub repo

Replace the URL with your own repo after you push these files.

In [ ]:
!git clone https://github.com/YOUR_NAME/YOUR_REPO.git
%cd YOUR_REPO

## 2. Install dependencies

In [ ]:
!pip uninstall -y torchao timm
!pip install -r requirements-colab.txt

## 3. Login to Hugging Face and W&B

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

import wandb
wandb.login()

## 3.5 Mount Google Drive

Use this to back up checkpoints and local W&B logs after training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Run GRPO training

This default run is intentionally small for Colab simulation.

In [ ]:
!accelerate launch --config_file configs/accelerate_config.yaml train_grpo.py \
  --model_name_or_path Qwen/Qwen2.5-1.5B-Instruct \
  --output_dir outputs/qwen2.5-1.5b-gsm8k-grpo \
  --max_samples 256 \
  --max_steps 50 \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 4 \
  --num_generations 2 \
  --fp16 true \
  --bf16 false \
  --use_lora true \
  --report_to wandb \
  --run_name colab-qwen25-15b-gsm8k-grpo

## 5. Evaluate a small sample

In [ ]:
!python eval_gsm8k.py \
  --model_name_or_path outputs/qwen2.5-1.5b-gsm8k-grpo \
  --max_samples 100

## 6. Back up results to Google Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/grpo-qwen25-gsm8k-finetune
!tar -czf /content/drive/MyDrive/grpo-qwen25-gsm8k-finetune/grpo-results-$(date +%Y%m%d-%H%M%S).tar.gz outputs wandb

## Optional: larger run

Use this after the small run succeeds and you have enough GPU memory.

In [ ]:
# !accelerate launch train_grpo.py \
#   --model_name_or_path Qwen/Qwen2.5-1.5B-Instruct \
#   --output_dir outputs/qwen2.5-1.5b-gsm8k-grpo-200step \
#   --max_samples 2000 \
#   --max_steps 200 \
#   --per_device_train_batch_size 2 \
#   --gradient_accumulation_steps 8 \
#   --num_generations 2 \
#   --fp16 true \
#   --use_lora true \
#   --report_to wandb \
#   --run_name qwen25-15b-gsm8k-grpo-200step